# Build NSW Crash Traffic-Volume Enriched Dataset

This notebook creates the enriched dataset used for RQ4. It combines:

1. `nsw_crash_preprocessed.xlsx`
2. `road_traffic_counts_station_reference.csv`
3. `road_traffic_counts_yearly_summary.csv`

The output is an analysis-ready crash dataset with matched traffic-volume features.

In [2]:
from __future__ import annotations

import json
import re
import zipfile
from pathlib import Path
from xml.etree.ElementTree import iterparse

import numpy as np
import pandas as pd

## 1. File Paths

If running this notebook on another computer, place the three source files in the same folder as this notebook, or update the paths below.

In [5]:
BASE_DIR = Path.cwd()
DOWNLOADS_DIR = Path.home() / "Downloads"

def first_existing(*paths):
    for path in paths:
        path = Path(path)
        if path.exists():
            return path
    return Path(paths[0])

CRASH_XLSX = first_existing(
    BASE_DIR / "nsw_crash_preprocessed.xlsx",
    DOWNLOADS_DIR / "nsw_crash_preprocessed.xlsx",
)

STATION_REFERENCE_CSV = first_existing(
    BASE_DIR / "road_traffic_counts_station_reference.csv",
    DOWNLOADS_DIR / "road_traffic_counts_station_reference.csv",
)

YEARLY_SUMMARY_CSV = first_existing(
    BASE_DIR / "road_traffic_counts_yearly_summary.csv",
    DOWNLOADS_DIR / "road_traffic_counts_yearly_summary.csv",
)

OUT_DIR = BASE_DIR / "traffic_enriched_dataset"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Crash file:", CRASH_XLSX)
print("Station reference:", STATION_REFERENCE_CSV)
print("Yearly summary:", YEARLY_SUMMARY_CSV)
print("Output folder:", OUT_DIR)

Crash file: /Users/alanjoshi/Downloads/nsw_crash_preprocessed.xlsx
Station reference: /Users/alanjoshi/Downloads/road_traffic_counts_station_reference.csv
Yearly summary: /Users/alanjoshi/Downloads/road_traffic_counts_yearly_summary.csv
Output folder: /Users/alanjoshi/Downloads/traffic_enriched_dataset


## 2. Helper Functions

In [8]:
def local_name(tag: str) -> str:
    return tag.rsplit("}", 1)[-1]


def clean_road_name(value: object) -> str:
    if pd.isna(value):
        return ""
    text = str(value).upper()
    text = re.sub(r"[^A-Z0-9 ]+", " ", text)
    words = [w for w in text.split() if w not in {"THE"}]
    return " ".join(words)


def load_crash_workbook(path: Path) -> pd.DataFrame:
    """Read the crash XLSX by streaming worksheet XML directly.

    This is used because the workbook metadata can be malformed for some
    Excel readers, even though the worksheet data itself is valid.
    """
    with zipfile.ZipFile(path) as zf:
        shared_strings = []
        for _, elem in iterparse(zf.open("xl/sharedStrings.xml"), events=("end",)):
            if local_name(elem.tag) == "si":
                parts = []
                for node in elem.iter():
                    if local_name(node.tag) == "t" and node.text:
                        parts.append(node.text)
                shared_strings.append("".join(parts))
                elem.clear()

        headers = None
        rows = []
        for _, elem in iterparse(zf.open("xl/worksheets/sheet1.xml"), events=("end",)):
            if local_name(elem.tag) != "row":
                continue

            cells = {}
            for cell in [node for node in elem if local_name(node.tag) == "c"]:
                ref = cell.attrib.get("r", "")
                col = "".join(re.findall("[A-Z]+", ref))
                col_idx = 0
                for char in col:
                    col_idx = col_idx * 26 + ord(char) - 64

                value_node = next((node for node in cell if local_name(node.tag) == "v"), None)
                value = None
                if value_node is not None:
                    raw = value_node.text
                    if cell.attrib.get("t") == "s" and raw is not None:
                        value = shared_strings[int(raw)]
                    else:
                        try:
                            value = float(raw) if raw and "." in raw else int(raw) if raw else None
                        except ValueError:
                            value = raw
                cells[col_idx] = value

            if cells:
                if headers is None:
                    headers = [str(cells.get(i, "")) for i in range(1, max(cells) + 1)]
                else:
                    rows.append({headers[i - 1]: cells.get(i) for i in range(1, len(headers) + 1)})
            elem.clear()

    return pd.DataFrame(rows)


def haversine_km(lat1: float, lon1: float, lat2: np.ndarray, lon2: np.ndarray) -> np.ndarray:
    radius = 6371.0088
    phi1 = np.radians(lat1)
    phi2 = np.radians(lat2)
    d_phi = np.radians(lat2 - lat1)
    d_lam = np.radians(lon2 - lon1)
    a = np.sin(d_phi / 2) ** 2 + np.cos(phi1) * np.cos(phi2) * np.sin(d_lam / 2) ** 2
    return 2 * radius * np.arcsin(np.sqrt(a))

## 3. Load Source Data

In [11]:
crashes = load_crash_workbook(CRASH_XLSX)
stations = pd.read_csv(STATION_REFERENCE_CSV, low_memory=False)
yearly = pd.read_csv(YEARLY_SUMMARY_CSV, low_memory=False)

print("Crashes:", crashes.shape)
print("Stations:", stations.shape)
print("Yearly summary:", yearly.shape)
crashes.head()

Crashes: (92189, 55)
Stations: (1783, 42)
Yearly summary: (274525, 27)


,Degree of crash,Degree of crash - detailed,Reporting year,Year of crash,Month of crash,Day of week of crash,Two-hour intervals,Street of crash,Street type,Distance,...,Total_casualties,Severity_score,Month_num,Day_num,Speed_limit_num,Is_fatal,Is_weekend,Is_dark,Is_wet,Crash_severity_encoded
0,Fatal,Fatal,2020,2020,January,Friday,22:00 - Midnight,HUME,HWY,630,...,1,4,1,5,110,1,0,1,0,2
1,Fatal,Fatal,2020,2020,January,Friday,22:00 - Midnight,PACIFIC,HWY,10,...,2,6,1,5,100,1,0,1,0,2
2,Fatal,Fatal,2020,2020,January,Friday,16:00 - 17:59,GOULBURN,RD,1150,...,1,4,1,5,100,1,0,0,0,2
3,Fatal,Fatal,2020,2020,January,Sunday,22:00 - Midnight,MARSDEN,RD,5,...,2,6,1,7,60,1,1,1,0,2
4,Fatal,Fatal,2020,2020,January,Wednesday,02:00 - 03:59,SCOTT,ST,50,...,1,4,1,3,40,1,0,1,0,2


## 4. Build Station-Year Traffic Features

The yearly traffic summary has multiple rows per station, year, period, vehicle class, and direction. This section pivots it into useful modelling columns such as all-day traffic and peak-period traffic.

In [14]:
def directional_value(group: pd.DataFrame) -> float:
    both = group[group["cardinal_direction_name"].astype(str).str.upper() == "BOTH"]
    if not both.empty:
        return float(both["traffic_count"].max())
    return float(group.groupby("cardinal_direction_name")["traffic_count"].max().sum())


def build_station_year_features(yearly: pd.DataFrame) -> pd.DataFrame:
    yearly = yearly.copy()
    yearly["traffic_count"] = pd.to_numeric(yearly["traffic_count"], errors="coerce")
    yearly = yearly.dropna(subset=["station_key", "year", "traffic_count"])
    yearly["classification_type"] = yearly["classification_type"].fillna("").str.upper()
    yearly["period"] = yearly["period"].fillna("").str.upper()

    wanted_periods = {
        "ALL DAYS": "all_days",
        "WEEKDAYS": "weekdays",
        "WEEKENDS": "weekends",
        "AM PEAK": "am_peak",
        "PM PEAK": "pm_peak",
    }
    wanted_classes = {
        "ALL VEHICLES": "all_vehicles",
        "UNCLASSIFIED": "unclassified",
        "LIGHT VEHICLES": "light_vehicles",
        "HEAVY VEHICLES": "heavy_vehicles",
    }

    filtered = yearly[
        yearly["period"].isin(wanted_periods)
        & yearly["classification_type"].isin(wanted_classes)
    ]

    rows = []
    group_cols = ["station_key", "year", "period", "classification_type"]
    for keys, group in filtered.groupby(group_cols):
        station_key, year, period, class_type = keys
        rows.append(
            {
                "station_key": station_key,
                "traffic_year": int(year),
                "feature": f"{wanted_classes[class_type]}_{wanted_periods[period]}",
                "value": directional_value(group),
            }
        )

    long = pd.DataFrame(rows)
    wide = long.pivot_table(
        index=["station_key", "traffic_year"],
        columns="feature",
        values="value",
        aggfunc="max",
    ).reset_index()
    wide.columns.name = None

    if "all_vehicles_all_days" not in wide and "unclassified_all_days" in wide:
        wide["all_vehicles_all_days"] = wide["unclassified_all_days"]
    else:
        wide["all_vehicles_all_days"] = wide.get("all_vehicles_all_days").fillna(wide.get("unclassified_all_days"))

    if "heavy_vehicles_all_days" in wide:
        wide["heavy_vehicle_share_all_days"] = wide["heavy_vehicles_all_days"] / wide["all_vehicles_all_days"]
    else:
        wide["heavy_vehicle_share_all_days"] = np.nan

    return wide


station_year_features = build_station_year_features(yearly)
station_year_features.to_csv(OUT_DIR / "traffic_station_year_features.csv", index=False)

print(station_year_features.shape)
station_year_features.head()

(8375, 23)


,station_key,traffic_year,all_vehicles_all_days,all_vehicles_am_peak,all_vehicles_pm_peak,all_vehicles_weekdays,all_vehicles_weekends,heavy_vehicles_all_days,heavy_vehicles_am_peak,heavy_vehicles_pm_peak,...,light_vehicles_am_peak,light_vehicles_pm_peak,light_vehicles_weekdays,light_vehicles_weekends,unclassified_all_days,unclassified_am_peak,unclassified_pm_peak,unclassified_weekdays,unclassified_weekends,heavy_vehicle_share_all_days
0,-1,2019,295.0,121.0,63.0,328.0,207.0,76.0,31.0,16.0,...,90.0,47.0,242.0,152.0,NaN,NaN,NaN,NaN,NaN,0.257627
1,55302,2007,44968.0,11262.0,11914.0,45884.0,44052.0,8538.0,2410.0,1936.0,...,8852.0,9978.0,35532.0,40414.0,NaN,NaN,NaN,NaN,NaN,0.189868
2,55304,2006,161514.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,161514.0,48730.0,48008.0,170448.0,139076.0,NaN
3,55304,2007,168858.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,168858.0,50354.0,50030.0,178658.0,144328.0,NaN
4,55304,2008,166592.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,166592.0,50912.0,49788.0,176058.0,138720.0,NaN


## 5. Match Crashes to Traffic Count Stations

Matching priority:

1. Same road and same LGA, nearest station.
2. Same road, nearest station.
3. Any road, nearest station.

The distance is calculated using latitude and longitude.

In [17]:
def match_crashes_to_stations(crashes: pd.DataFrame, stations: pd.DataFrame) -> pd.DataFrame:
    station_cols = [
        "station_key",
        "station_id",
        "name",
        "road_name",
        "road_name_base",
        "road_name_type",
        "lga",
        "suburb",
        "wgs84_latitude",
        "wgs84_longitude",
        "road_classification_admin",
        "road_functional_hierarchy",
        "lane_count",
        "quality_rating",
    ]
    stations = stations[station_cols].dropna(subset=["wgs84_latitude", "wgs84_longitude"]).copy()
    stations["match_road"] = stations["road_name_base"].map(clean_road_name)
    stations["match_lga"] = stations["lga"].map(clean_road_name)

    by_road = {road: frame for road, frame in stations.groupby("match_road") if road}
    all_lats = stations["wgs84_latitude"].to_numpy(float)
    all_lons = stations["wgs84_longitude"].to_numpy(float)

    match_rows = []
    for idx, row in crashes.iterrows():
        lat = row.get("Latitude")
        lon = row.get("Longitude")
        crash_road = clean_road_name(row.get("Street of crash"))
        crash_lga = clean_road_name(row.get("LGA"))

        if pd.isna(lat) or pd.isna(lon):
            match_rows.append({"crash_row_id": idx, "match_method": "no_coordinates"})
            continue

        candidates = by_road.get(crash_road)
        method = "same_road_nearest_station"
        if candidates is None or candidates.empty:
            candidates = stations
            method = "nearest_station_any_road"
        elif crash_lga:
            same_lga = candidates[candidates["match_lga"] == crash_lga]
            if not same_lga.empty:
                candidates = same_lga
                method = "same_road_same_lga_nearest_station"

        if candidates is stations:
            distances = haversine_km(float(lat), float(lon), all_lats, all_lons)
            pos = int(np.nanargmin(distances))
            nearest = stations.iloc[pos]
            distance = float(distances[pos])
        else:
            cand_lats = candidates["wgs84_latitude"].to_numpy(float)
            cand_lons = candidates["wgs84_longitude"].to_numpy(float)
            distances = haversine_km(float(lat), float(lon), cand_lats, cand_lons)
            pos = int(np.nanargmin(distances))
            nearest = candidates.iloc[pos]
            distance = float(distances[pos])

        match_rows.append(
            {
                "crash_row_id": idx,
                "matched_station_key": nearest["station_key"],
                "matched_station_id": nearest["station_id"],
                "matched_station_name": nearest["name"],
                "matched_station_road_name": nearest["road_name"],
                "matched_station_lga": nearest["lga"],
                "matched_station_suburb": nearest["suburb"],
                "matched_station_latitude": nearest["wgs84_latitude"],
                "matched_station_longitude": nearest["wgs84_longitude"],
                "matched_station_road_classification": nearest["road_classification_admin"],
                "matched_station_road_hierarchy": nearest["road_functional_hierarchy"],
                "matched_station_lane_count": nearest["lane_count"],
                "matched_station_quality_rating": nearest["quality_rating"],
                "matched_station_distance_km": distance,
                "match_method": method,
            }
        )

    return pd.DataFrame(match_rows)


matches = match_crashes_to_stations(crashes, stations)
print(matches.shape)
matches.head()

(92189, 15)


,crash_row_id,matched_station_key,matched_station_id,matched_station_name,matched_station_road_name,matched_station_lga,matched_station_suburb,matched_station_latitude,matched_station_longitude,matched_station_road_classification,matched_station_road_hierarchy,matched_station_lane_count,matched_station_quality_rating,matched_station_distance_km,match_method
0,0,15216011,6136,Hume Highway,Hume Highway,Gundagai,Coolac,-34.905838,148.185486,State,Primary Road,TwoOrMoreLanes,5,10.625783,same_road_nearest_station
1,1,55437,04002,Pacific Highway,Pacific Highway,Clarence Valley,Clarenza,-29.737230,152.980667,State,Primary Road,TwoOrMoreLanes,5,1.571353,same_road_same_lga_nearest_station
2,2,58121,94296,Goulburn Street,Goulburn Street,Upper Lachlan,Crookwell,-34.464546,149.478699,State,Arterial Road,TwoOrMoreLanes,5,11.012887,same_road_same_lga_nearest_station
3,3,57048,50232,Marsden Road,Marsden Road,Parramatta,Ermington,-33.802933,151.066223,State,Arterial Road,TwoOrMoreLanes,4,0.718728,same_road_nearest_station
4,4,57790,92653,Scott Road,Scott Road,Tamworth Regional,South Tamworth,-31.109468,150.929977,State,Primary Road,TwoOrMoreLanes,5,217.465726,same_road_nearest_station


## 6. Add Traffic Counts to Crash Rows

The first join uses the exact crash year. If an exact station-year traffic count is not available, the notebook uses the nearest previous traffic-count year for the same station. This is recorded in `traffic_year_match_type`.

In [20]:
def add_fallback_traffic_year(enriched: pd.DataFrame, station_year_features: pd.DataFrame) -> pd.DataFrame:
    feature_cols = [col for col in station_year_features.columns if col not in {"station_key", "traffic_year"}]
    by_station = {
        station_key: frame.sort_values("traffic_year").reset_index(drop=True)
        for station_key, frame in station_year_features.groupby("station_key")
    }

    enriched["traffic_year_used"] = enriched["traffic_year"]
    enriched["traffic_year_match_type"] = np.where(
        enriched["all_vehicles_all_days"].notna(), "exact_year", "missing"
    )

    missing = enriched["all_vehicles_all_days"].isna() & enriched["matched_station_key"].notna()
    for idx, row in enriched.loc[missing].iterrows():
        station_key = row["matched_station_key"]
        crash_year = row["traffic_year"]
        frame = by_station.get(station_key)
        if frame is None or frame.empty or pd.isna(crash_year):
            continue

        years = frame["traffic_year"].to_numpy(dtype=float)
        prior_positions = np.flatnonzero(years <= float(crash_year))
        if len(prior_positions):
            chosen_pos = prior_positions[np.argmax(years[prior_positions])]
            match_type = "fallback_prior_year"
        else:
            chosen_pos = int(np.argmin(np.abs(years - float(crash_year))))
            match_type = "fallback_nearest_future_year"

        chosen = frame.iloc[int(chosen_pos)]
        for col in feature_cols:
            enriched.at[idx, col] = chosen[col]
        enriched.at[idx, "traffic_year_used"] = int(chosen["traffic_year"])
        enriched.at[idx, "traffic_year_match_type"] = match_type

    return enriched


crashes_with_id = crashes.reset_index().rename(columns={"index": "crash_row_id"})
enriched = crashes_with_id.merge(matches, on="crash_row_id", how="left")
enriched["traffic_year"] = pd.to_numeric(enriched["Year of crash"], errors="coerce").astype("Int64")
enriched = enriched.merge(
    station_year_features.rename(columns={"station_key": "matched_station_key"}),
    on=["matched_station_key", "traffic_year"],
    how="left",
)
enriched = add_fallback_traffic_year(enriched, station_year_features)

print(enriched.shape)
enriched[["Street of crash", "LGA", "matched_station_road_name", "matched_station_distance_km", "all_vehicles_all_days", "traffic_year_match_type"]].head()

(92189, 94)


,Street of crash,LGA,matched_station_road_name,matched_station_distance_km,all_vehicles_all_days,traffic_year_match_type
0,HUME,Cootamundra-Gundagai,Hume Highway,10.625783,4153.0,exact_year
1,PACIFIC,Clarence Valley,Pacific Highway,1.571353,6278.0,fallback_prior_year
2,GOULBURN,Upper Lachlan,Goulburn Street,11.012887,3174.0,fallback_prior_year
3,MARSDEN,Ryde,Marsden Road,0.718728,11230.0,exact_year
4,SCOTT,Newcastle,Scott Road,217.465726,26186.0,fallback_prior_year


## 7. Add Match Quality and Create Analysis-Ready Dataset

The analysis-ready dataset keeps high and medium quality traffic matches only.

In [23]:
def add_match_quality(enriched: pd.DataFrame) -> pd.DataFrame:
    distance = enriched["matched_station_distance_km"]
    method = enriched["match_method"].fillna("")
    high = method.eq("same_road_same_lga_nearest_station") & distance.le(5)
    medium = (
        (method.eq("same_road_nearest_station") & distance.le(5))
        | (method.eq("nearest_station_any_road") & distance.le(1))
    )
    enriched["traffic_match_quality"] = np.select(
        [high, medium],
        ["high", "medium"],
        default="low",
    )
    return enriched


enriched = add_match_quality(enriched)

analysis_ready = enriched[
    enriched["traffic_match_quality"].isin(["high", "medium"])
    & enriched["all_vehicles_all_days"].notna()
].copy()

print("Full enriched rows:", len(enriched))
print("Analysis-ready rows:", len(analysis_ready))
enriched["traffic_match_quality"].value_counts(dropna=False)

Full enriched rows: 92189
Analysis-ready rows: 47092


traffic_match_quality
low       45096
medium    26855
high      20238
Name: count, dtype: int64

## 8. Reorder Important Columns and Save Outputs

In [26]:
key_cols = [
    "crash_row_id",
    "Year of crash",
    "Month of crash",
    "Day of week of crash",
    "Street of crash",
    "Street type",
    "Town",
    "LGA",
    "Latitude",
    "Longitude",
    "Degree of crash",
    "No. of traffic units involved",
    "Total_casualties",
    "matched_station_key",
    "matched_station_name",
    "matched_station_road_name",
    "matched_station_lga",
    "matched_station_suburb",
    "matched_station_distance_km",
    "match_method",
    "traffic_match_quality",
    "traffic_year",
    "traffic_year_used",
    "traffic_year_match_type",
    "all_vehicles_all_days",
    "all_vehicles_weekdays",
    "all_vehicles_weekends",
    "all_vehicles_am_peak",
    "all_vehicles_pm_peak",
    "heavy_vehicles_all_days",
    "heavy_vehicle_share_all_days",
    "matched_station_road_classification",
    "matched_station_road_hierarchy",
    "matched_station_lane_count",
    "matched_station_quality_rating",
]
key_cols = [col for col in key_cols if col in enriched.columns]
other_cols = [col for col in enriched.columns if col not in key_cols]

enriched = enriched[key_cols + other_cols]
analysis_ready = analysis_ready[[col for col in enriched.columns if col in analysis_ready.columns]]

enriched_path = OUT_DIR / "nsw_crash_enriched_with_traffic_volume.csv"
analysis_ready_path = OUT_DIR / "nsw_crash_traffic_volume_analysis_ready.csv"
features_path = OUT_DIR / "traffic_station_year_features.csv"

enriched.to_csv(enriched_path, index=False)
analysis_ready.to_csv(analysis_ready_path, index=False)
station_year_features.to_csv(features_path, index=False)

coverage = {
    "crash_rows": int(len(enriched)),
    "station_rows": int(len(stations)),
    "station_year_feature_rows": int(len(station_year_features)),
    "matched_to_station_rows": int(enriched["matched_station_key"].notna().sum()),
    "rows_with_all_vehicles_all_days": int(enriched["all_vehicles_all_days"].notna().sum()),
    "traffic_volume_coverage_rate": float(enriched["all_vehicles_all_days"].notna().mean()),
    "match_methods": enriched["match_method"].value_counts(dropna=False).to_dict(),
    "traffic_match_quality": enriched["traffic_match_quality"].value_counts(dropna=False).to_dict(),
    "analysis_ready_rows": int(len(analysis_ready)),
    "traffic_year_match_types": enriched["traffic_year_match_type"].value_counts(dropna=False).to_dict(),
    "median_station_distance_km": float(enriched["matched_station_distance_km"].median()),
    "p90_station_distance_km": float(enriched["matched_station_distance_km"].quantile(0.90)),
    "output_files": {
        "enriched_crash_dataset": str(enriched_path),
        "analysis_ready_dataset": str(analysis_ready_path),
        "station_year_features": str(features_path),
    },
}

(OUT_DIR / "enrichment_coverage_summary.json").write_text(json.dumps(coverage, indent=2), encoding="utf-8")
print(json.dumps(coverage, indent=2))

{
  "crash_rows": 92189,
  "station_rows": 1783,
  "station_year_feature_rows": 8375,
  "matched_to_station_rows": 92189,
  "rows_with_all_vehicles_all_days": 92183,
  "traffic_volume_coverage_rate": 0.9999349163132261,
  "match_methods": {
    "nearest_station_any_road": 44893,
    "same_road_same_lga_nearest_station": 25767,
    "same_road_nearest_station": 21529
  },
  "traffic_match_quality": {
    "low": 45096,
    "medium": 26855,
    "high": 20238
  },
  "analysis_ready_rows": 47092,
  "traffic_year_match_types": {
    "fallback_prior_year": 80884,
    "exact_year": 11305
  },
  "median_station_distance_km": 1.6532611667624135,
  "p90_station_distance_km": 19.848125541451285,
  "output_files": {
    "enriched_crash_dataset": "/Users/alanjoshi/Downloads/traffic_enriched_dataset/nsw_crash_enriched_with_traffic_volume.csv",
    "analysis_ready_dataset": "/Users/alanjoshi/Downloads/traffic_enriched_dataset/nsw_crash_traffic_volume_analysis_ready.csv",
    "station_year_features": "/

## 9. Output Files

The key file for modelling is:

`traffic_enriched_dataset/nsw_crash_traffic_volume_analysis_ready.csv`

This file is used in the RQ4 classification analysis notebook.